In [1]:
using DataFrames
using VegaLite
using LinearAlgebra
using XLSX

# Using Real Data (2 sectors)

Manufacturing and non-manufacturing goods

## Data Assembly

used Gemini to do the cleaning

In [2]:
# 1. Load excel data
xf = XLSX.readxlsx("./dataset.xlsx")

# 2. Load the 44x44 Bilateral Trade Matrix
# Based on the file structure, the numeric data starts at B2 (to skip headers/names)
# and extends 44 columns to AS and 44 rows to 45.
trade_04 = Float64.(xf["Trade"]["B2:AS45"])

# 3. Load GDP (44x1)
# The GDP snippet shows no header, so we take column B, rows 1 to 44.
gdp_04 = Float64.(xf["GDP"]["B1:B44"])

# 4. Load Current Account and G&S Balance (44x2)
# This sheet has headers, so we start at row 2. 
# Column B is Current Account, Column C is G&S Balance.
catb_04 = Float64.(xf["catb"]["B2:C45"])


44×2 Matrix{Float64}:
   11.12        7.64
    3.42616    11.7494
  -40.0654    -18.7085
    0.765       5.758
   16.0965     22.3824
   11.7376     28.9895
   21.1568     40.4222
    1.58637     8.5067
   84.3892     63.9936
   -0.93834    -0.33297
    ⋮        
  -54.8654    -39.8406
   27.485      29.0785
   56.5823     34.475
    6.85739     6.74828
  -15.604     -11.094
  -35.18      -63.96
 -665.29     -611.3
   13.83       17.804
  -55.3       198.0

In [3]:
α = 0.188
β = 0.312
θ = 8.28

# Bilateral trade and deficit in manufactures
xbilat = copy(trade_04') 

exports = vec(sum(xbilat, dims=1)) 
imports = vec(sum(xbilat, dims=2)) 
defm = imports .- exports          

# GDP data (Scaling to Billions)
gdp = gdp_04 ./ 1e9 

# Balance the world current account across all 44 entities
catb = copy(catb_04)
catbw = catb[44, :] 
catb[44, :] = catbw .- vec(sum(catb[1:43, :], dims=1)) 
catb = catb .- (catbw ./ 44)' 

casurplus = catb[:, 1] 
tsurplus = catb[:, 2]  

# Implied manufacturing values
xgdp = vec(gdp) .- vec(tsurplus)               
vm = α .* xgdp .- defm           
ym = vm ./ β                      
xnn = ym .- exports                  

# Spending matrix with home sales on diagonal
xnimatrix = xbilat - Diagonal(diag(xbilat)) + Diagonal(xnn)

# ---------------------------------------------------------
# Aggregation (The "aggmat" logic will now work perfectly)
# ---------------------------------------------------------
aggmat = Matrix{Float64}(I, 44, 44)
aggmat[5, 26] = 1.0   # BelLux + Netherlands
aggmat[18, 24] = 1.0  # Indonesia + Malaysia
aggmat[18, 34] = 1.0  # Indonesia + Singapore
aggmat[18, 39] = 1.0  # Indonesia + Thailand

rows_to_keep = setdiff(1:44, [24, 26, 34, 39])
aggmat = aggmat[rows_to_keep, :]

# Redefine everything for the final 40 countries
xnimatrix = aggmat * xnimatrix * aggmat' 
gdp = aggmat * gdp                      
casurplus = aggmat * casurplus          
tsurplus = aggmat * tsurplus            
deficitt = -tsurplus                      

xnn = diag(xnimatrix)                     
imports = vec(sum(xnimatrix, dims=2)) .- xnn
exports = vec(sum(xnimatrix, dims=1)) .- xnn
deficitm = imports .- exports             

xn = xnn .+ imports                       
pimatrix = xnimatrix ./ xn                

# Final Counterfactual targets
defmc = casurplus .+ deficitm              
deftc = defmc .+ deficitt .- deficitm

40-element Vector{Float64}:
    9.23681818181818
   -2.5664218181818175
  -15.600081818181813
    0.763818181818182
   12.36423636363638
  -11.495081818181816
  -13.508581818181817
   -1.163511818181819
   26.152418181818177
    5.151448181818182
    ⋮
   -0.07434181818181873
   -9.267981818181816
    4.163318181818184
   27.864118181818178
    1.2468181818181847
   34.536818181818205
  -48.23318181818183
    1.7828181818181825
 -130.51305181818188

Rename variables to the notation and object types I was using earlier

In [4]:
# 1. National Income (GDP)
Y = vec(gdp)

# 2. Expenditure Shares (pi_ni)
# Rows = Importers (n), Columns = Exporters (i)
pi_matrix = pimatrix

# 3. Counterfactual Manufacturing Deficit (D_M')
# This is the manufactured deficit required for a 0 current account balance
D_Mprime = vec(defmc)

# 4. Counterfactual Total Trade Deficit (D')
# Total trade imbalance required for a 0 current account balance
D_prime = vec(deftc)

# Verification of dimensions for a 40-country solver
println("Size of Y: ", size(Y))           # Expected: (40,)
println("Size of pi: ", size(pi_matrix))   # Expected: (40, 40)
println("Size of D_M': ", size(D_Mprime)) # Expected: (40,)

Size of Y: (40,)
Size of pi: (40, 40)
Size of D_M': (40,)


Now apply the solver from earlier

It didn't work. Getting "Exponentiation yielding a complex result requires a complex argument." Somehow had negative values for prices or wages?


In [5]:
#Iteration parameters 
tol = 1e-6
max_iter = 10000
step_size = 0.01

N = length(Y)

#Initial guesses for endogenous variables w_hat and p_hat
w_hat = ones(N)
p_hat = ones(N)

iteration_count = 0
error = 1.

while error > tol && iteration_count < max_iter

    #Inner Loop: Solve for prices (equation 7)
    p_error = 1
    while p_error > tol
        p_hat_update = zeros(N) #update guess of p_hat by filling in equation 7
        #need to calculate p_n for each N 
        for n in 1:N
            #Calculate term inside parantheses in equation 7
            term = sum([pi_matrix[n,k] * w_hat[k]^(-θ*β) * p_hat[k]^(-θ*(1-β)) for k in 1:N])
            p_hat_update[n] = term^(-1/θ)
        end
       
        p_error = maximum(abs.(p_hat_update - p_hat))
        p_hat = p_hat_update #norm(p_hat_update - p_hat) apparently this is better condition to guarantee convergence
    end

    #In the Outer Loop, solve Equation 6 
    #Equation 6 is a system of N equations. 1 equation for each country 
    LHS = zeros(N) #LHS is income/supply.
    RHS = zeros(N) #RHS is demand 

    for i in 1:N 
        #Calculate LHS 
        LHS[i] = w_hat[i]*Y[i] + D_prime[i] - (1/α)*D_Mprime[i] 

        #Calculate RHS
        sum_RHS = 0 
        for n in 1:N
            numerator = pi_matrix[n,i] * w_hat[i]^(-θ*β) * p_hat[i]^(-θ*(1-β))
            denominator = p_hat[n]^(-θ) #note the denominator is just an expression in terms of p_hat which we already have
            multiplicative_term = w_hat[n]*Y[n] + D_prime[n] - ((1-β)/α)*D_Mprime[n] 

            sum_RHS += (numerator/denominator)*multiplicative_term
        end
        RHS[i] = sum_RHS

    end

    #Now update wage. 
    # w_hat_update = zeros(N) #update guess of w_hat 
    # for i in 1:N
    #     #RHS (demand) > LHS (supply/income), increase wages so that LHS increases
    #     w_hat_update[i] = w_hat[i] * ((RHS[i]/LHS[i])^step_size) #RHS/LHS can't be negative or else will result in complex number
    # end
    #Now update wage. 
    w_hat_update = zeros(N) 
    for i in 1:N
        # Diagnostic check to see which country is failing
        if LHS[i] <= 0 || RHS[i] <= 0
            println("Warning: Iteration $iteration_count. Country $i has LHS = $(LHS[i]), RHS = $(RHS[i])")
        end
        
        # Safe update: binds LHS and RHS to a tiny positive number to prevent complex errors
        safe_LHS = max(LHS[i], 1e-12)
        safe_RHS = max(RHS[i], 1e-12)
        
        w_hat_update[i] = w_hat[i] * ((safe_RHS/safe_LHS)^step_size)
    end

    #Need to keep GDP (Y) constant 
    # current_Y = sum(w_hat_update .* Y)
    # initial_Y = sum(Y)
    # norm_factor = initial_Y/current_Y

    # w_hat = w_hat_update*norm_factor

    #Use USA to rescale everything
    w_hat = w_hat_update ./ w_hat_update[38]

    #check for convergence
    error = maximum(abs.((RHS .- LHS) ./ LHS)) #norm(RHS - LHS) 
    iteration_count += 1
end

println("Convergence achieved in ", iteration_count, " iterations.")
println("Final w_hat:", w_hat)
println("Final p_hat:", p_hat)

Convergence achieved in 735 iterations.
Final w_hat:[1.570656185679939, 1.114934359687092, 0.9653097378799028, 1.0836790954342115, 1.206614725415464, 1.0956754982967005, 1.0732860372710935, 1.1041500132250661, 1.0996035789520893, 1.0718156825491976, 1.1130261777098807, 1.171514225581594, 1.1284950242038374, 1.075819496727699, 1.1068851860457194, 0.9408973618271915, 1.0895375236285998, 1.1561863815128457, 1.0796615666688119, 1.094937977770989, 1.0726919218194169, 1.113260833516283, 1.1016517824881993, 1.03923357084411, 0.9698472168232586, 1.3886706661380535, 1.0774130002066684, 1.0899206105039743, 1.093839324038833, 1.0067457218939424, 1.2224129468111997, 1.0485270339019868, 1.0190203101627848, 1.1483203808555815, 1.2658115875574285, 1.024056210756821, 1.0538416806811801, 1.0, 1.3587152353902467, 1.0915016915391604]
Final p_hat:[1.1139848870763522, 1.0900274537771764, 1.022521559695877, 1.0881901358069115, 1.082741396851959, 1.0873722370226, 1.0496890940600176, 1.0844359651840008, 1.091

In [6]:
# 1. Calculate Welfare Change
Y_prime = w_hat .* Y
D_total_initial = deftc .- casurplus

welfare_hat = [
    (w_hat[i]/p_hat[i])^α * (1 + D_prime[i]/Y_prime[i]) / (1 + D_total_initial[i]/Y[i]) for i in 1:N
    ]

# 2. Define the 40 Aggregated Country Names
# These match the setdiff(1:44, [24, 26, 34, 39]) logic
country_names = [
    "Algeria", "Argentina", "Australia", "Austria", "BelLux-Netherlands",
    "Brazil", "Canada", "Chile", "China-HK", "Colombia", "Denmark", "Egypt",
    "Finland", "France", "Germany", "Greece", "India", 
    "Indonesia-ASEAN", "Ireland", "Israel", "Italy", "Japan", "Korea", 
    "Mexico", "New Zealand", "Norway", "Pakistan", "Peru", "Philippines", 
    "Portugal", "Russia", "South Africa", "Spain", "Sweden", "Switzerland", 
    "Turkey", "UK", "USA", "Venezuela", "ROW"
]

# 3. Print LaTeX Table Rows
println("\\begin{table}[h]")
println("\\centering")
println("\\begin{tabular}{l|ccc}")
println("\\hline")
println("Country & \$\\hat{w}\$ & \$\\hat{p}\$ & \$\\hat{W}\$ \\\\")
println("\\hline")

for i in 1:length(welfare_hat)
    # Format to 4 decimal places
    w = round(w_hat[i], digits=4)
    p = round(p_hat[i], digits=4)
    wf = round(welfare_hat[i], digits=4)
    
    println("$(country_names[i]) & $w & $p & $wf \\\\")
end

println("\\hline")
println("\\end{tabular}")
println("\\caption{Counterfactual Results: Change in Wages, Prices, and Welfare}")
println("\\end{table}")

\begin{table}[h]
\centering
\begin{tabular}{l|ccc}
\hline
Country & $\hat{w}$ & $\hat{p}$ & $\hat{W}$ \\
\hline
Algeria & 1.5707 & 1.114 & 1.1842 \\
Argentina & 1.1149 & 1.09 & 1.0383 \\
Australia & 0.9653 & 1.0225 & 0.9303 \\
Austria & 1.0837 & 1.0882 & 1.006 \\
BelLux-Netherlands & 1.2066 & 1.0827 & 1.1008 \\
Brazil & 1.0957 & 1.0874 & 1.0256 \\
Canada & 1.0733 & 1.0497 & 1.0291 \\
Chile & 1.1042 & 1.0844 & 1.0359 \\
China-HK & 1.0996 & 1.0912 & 1.0424 \\
Colombia & 1.0718 & 1.0693 & 1.0002 \\
Denmark & 1.113 & 1.0988 & 1.0336 \\
Egypt & 1.1715 & 1.135 & 1.0505 \\
Finland & 1.1285 & 1.1086 & 1.0612 \\
France & 1.0758 & 1.079 & 0.9969 \\
Germany & 1.1069 & 1.0935 & 1.0429 \\
Greece & 0.9409 & 1.0478 & 0.9273 \\
India & 1.0895 & 1.0881 & 1.0089 \\
Indonesia-ASEAN & 1.1562 & 1.0881 & 1.1057 \\
Ireland & 1.0797 & 1.0758 & 1.0118 \\
Israel & 1.0949 & 1.0839 & 1.0324 \\
Italy & 1.0727 & 1.0799 & 0.991 \\
Japan & 1.1133 & 1.1044 & 1.0379 \\
Korea & 1.1017 & 1.0906 & 1.0463 \\
Mexico & 1.039

This test code below was fully written by Gemini

In [7]:
# ---------------------------------------------------------
# ACID TEST: VERIFYING BALANCED TRADE
# ---------------------------------------------------------

# 1. Reconstruct Counterfactual Trade Shares (pi_prime)
pi_prime = zeros(N, N)
for n in 1:N, i in 1:N
    numerator = pi_matrix[n,i] * w_hat[i]^(-θ*β) * p_hat[i]^(-θ*(1-β))
    denominator = p_hat[n]^(-θ)
    pi_prime[n,i] = numerator / denominator
end

# 2. Reconstruct Counterfactual Total Manufacturing Expenditure (X_M_prime)
# This matches the LHS of your equation 6
X_M_prime = zeros(N)
for n in 1:N
    X_M_prime[n] = (α/β) * (w_hat[n]*Y[n] + D_prime[n]) - ((1-β)/β) * D_Mprime[n]
end

# 3. Calculate Counterfactual Bilateral Trade Flows (X_ni_prime)
X_ni_prime = pi_prime .* X_M_prime

# 4. Calculate Final Counterfactual Manufacturing Deficit
imports_M_prime = vec(sum(X_ni_prime, dims=2)) .- diag(X_ni_prime)
exports_M_prime = vec(sum(X_ni_prime, dims=1)) .- diag(X_ni_prime)
actual_D_M_prime = imports_M_prime .- exports_M_prime

# ---------------------------------------------------------
# THE FINAL CHECKS
# ---------------------------------------------------------

# Check 1: Did the solver hit the manufacturing target?
mfg_target_error = maximum(abs.(actual_D_M_prime .- D_Mprime))
println("Max error in manufacturing target: ", mfg_target_error)

# Check 2: IS THE WORLD'S CURRENT ACCOUNT ZERO?
# Since CA' = -(D_Total') = -(D_M' + D_NM')
# And your code's D_prime variable is exactly equal to -D_NM'
actual_CA_prime = D_prime .- actual_D_M_prime

println("Max deviation from 0 Current Account: ", maximum(abs.(actual_CA_prime)))

Max error in manufacturing target: 0.0037525171861716444
Max deviation from 0 Current Account: 187.29394923764562
